# calculate_pr — Orquestador del test PR

Lee las plantas activas del Excel de configuración (o acepta una lista explícita)
y lanza `pr_runner` para cada una.

**Parámetros (sobreescribibles por el pipeline):**
- `plant_codes` — JSON array o lista separada por comas. Vacío = todas las plantas activas del Excel.
- `period_start` / `period_end` — ISO datetime string. Vacío = máximo rango disponible en `scada_proc` y `proc_pvsyst`.
- `run_id` — identificador del run. Vacío = timestamp automático.

In [ ]:
plant_codes  = ""  
period_start = ""   
period_end   = ""   
run_id       = ""   

In [ ]:
import json
import datetime as dt
from pathlib import Path
import pandas as pd
import notebookutils

ABFS_BASE       = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"
ABFS_CONFIG_DIR = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG'
CONFIG_XLSX_TMP = '/tmp/plants_config_pr.xlsx'


def _download_if_missing(abfs_path, local_path):
    if not Path(local_path).exists():
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local_path, 'wb') as f:
            f.write(row['content'])


# run_id automatico
if not run_id:
    run_id = f"run_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"run_id: {run_id}")


# Lista de plantas
if plant_codes:
    try:
        plants = json.loads(plant_codes)
    except (json.JSONDecodeError, ValueError):
        plants = [p.strip() for p in plant_codes.split(',') if p.strip()]
else:
    _download_if_missing(f'{ABFS_CONFIG_DIR}/plants_config.xlsx', CONFIG_XLSX_TMP)
    df = pd.read_excel(CONFIG_XLSX_TMP, sheet_name='01_Plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(
        ['TRUE', 'VERDADERO', '1', 'SI', 'SÍ']
    )]
    plants = df['plant_code'].dropna().tolist()

print(f"Plantas a procesar ({len(plants)}): {plants}")


# Rango de fechas: maximo disponible en las tablas
if not period_start or not period_end:
    plant_filter = "'" + "','".join(plants) + "'"

    scada_row  = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code IN ({plant_filter})
    """).first()

    pvsyst_row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM proc_pvsyst
        WHERE plant_code IN ({plant_filter})
    """).first()

    all_mins = [d for d in (scada_row['mn'], pvsyst_row['mn']) if d is not None]
    all_maxs = [d for d in (scada_row['mx'], pvsyst_row['mx']) if d is not None]

    if not all_mins or not all_maxs:
        raise ValueError("No hay datos en scada_proc ni proc_pvsyst para las plantas seleccionadas.")

    period_start = str(min(all_mins))
    period_end   = str(max(all_maxs))

print(f"Periodo: {period_start}  ->  {period_end}")

In [ ]:
# Ejecutar pr_runner por planta
results = []

for pc in plants:
    print(f"\n{'-'*55}")
    print(f">> {pc}")
    try:
        out = notebookutils.notebook.run(
            'pr_runner',
            3600,
            {
                'plant_code':   pc,
                'period_start': period_start,
                'period_end':   period_end,
                'run_id':       run_id,
            }
        )
        r = json.loads(out)
        results.append({'plant_code': pc, 'status': 'ok', **r})
        print(f"   {r.get('verdict','?')}  Delta={r.get('delta_pct','?')}%")
    except Exception as e:
        msg = str(e)[:400]
        results.append({'plant_code': pc, 'status': 'failed', 'error': msg})
        print(f"   FAILED: {msg}")


# Resumen
n_ok     = sum(1 for r in results if r['status'] == 'ok')
n_failed = sum(1 for r in results if r['status'] == 'failed')

print(f"\n{'='*55}")
print(f"Resumen: {n_ok}/{len(results)} OK  |  {n_failed} fallidos")
print(f"{'='*55}")
for r in results:
    if r['status'] == 'ok':
        print(f"  OK   {r['plant_code']:<12} {r.get('verdict','?'):4}  Delta={r.get('delta_pct','?')}%")
    else:
        print(f"  FAIL {r['plant_code']:<12} {r.get('error','')[:80]}")

In [ ]:
notebookutils.notebook.exit(json.dumps({
    'plants_processed': len(results),
    'ok':               n_ok,
    'failed':           n_failed,
    'results':          results,
}))